# ClonField Studio · Colab
Escolha um runtime com GPU e execute na ordem. Cadastre `CIVITAI_TOKEN` e `HF_TOKEN` no painel Secrets do Colab e autorize acesso. O código vem do GitHub e os arquivos autenticados ficam no runtime ou na pasta do Drive escolhida.

A última célula imprime o link **STUDIO** (o app novo, com Apps, Modelos, LoRAs, Histórico e Sistema) e a senha da sessão. O Gradio antigo continua disponível no link público como alternativa.

O Studio detecta sua GPU e ajusta resolução, qualidade e flags do ComfyUI. Os modelos baixam sob demanda, com progresso e retomada.


In [ ]:
# Atualizar o código público
from pathlib import Path
import subprocess, sys

origin = "https://github.com/Vit0lze/clonfield-studio.git"
repo = Path("/content/clonfield_runtime/clonfield-studio")
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    current_origin = subprocess.check_output(["git", "-C", str(repo), "remote", "get-url", "origin"], text=True).strip()
    if current_origin != origin:
        raise RuntimeError("O diretório existente aponta para outro repositório.")
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", origin, str(repo)], check=True)


In [ ]:
# 1. Instalar dependências. Só instala o que falta.
import subprocess, sys
from importlib import metadata
wanted = {'gradio': 'gradio==5.49.1', 'websockets': 'websockets>=13,<16', 'huggingface_hub': 'huggingface_hub',
          'pillow': 'Pillow==11.3.0', 'requests': 'requests', 'spandrel': 'spandrel', 'numpy': 'numpy',
          'uvicorn': 'uvicorn', 'python-multipart': 'python-multipart', 'safetensors': 'safetensors',
          'jsonschema': 'jsonschema', 'pyyaml': 'pyyaml', 'num2words': 'num2words', 'onnxruntime': 'onnxruntime',
          'psd-tools': 'psd-tools', 'kokoro-onnx': 'kokoro-onnx', 'faster-whisper': 'faster-whisper', 'soundfile': 'soundfile'}
missing = []
for name, spec in wanted.items():
    try:
        version = metadata.version(name)
        if '==' in spec and version != spec.split('==')[1]:
            missing.append(spec)
    except metadata.PackageNotFoundError:
        missing.append(spec)
if missing:
    print('Instalando:', ', '.join(missing), flush=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *missing])
else:
    print('Dependências já instaladas.', flush=True)


In [ ]:
# Credenciais opcionais do Colab. Não escreva tokens no notebook.
# Crie Secrets no painel lateral (ícone de chave) e permita acesso: HF_TOKEN e CIVITAI_TOKEN. O chat usa um modelo local (célula do LLM local, abaixo) e não precisa de chave; jev (ou TYPESAFE_API_TOKEN) só vale se você ligar CLONFIELD_JEV=1.
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None
# Cada credencial aceita mais de um nome de Secret; vale o primeiro que existir.
SECRETS = {'HF_TOKEN': ('HF_TOKEN',), 'CIVITAI_TOKEN': ('CIVITAI_TOKEN',),
           'TYPESAFE_API_TOKEN': ('TYPESAFE_API_TOKEN', 'JEV_API_KEY', 'jev')}
for env_name, secret_names in SECRETS.items():
    value = None
    for secret_name in secret_names:
        if userdata is None:
            break
        try:
            value = userdata.get(secret_name)
        except Exception:
            value = None
        if value:
            break
    if value:
        os.environ[env_name] = value
        print(env_name + ': configurado para esta sessão')
    elif os.environ.get(env_name):
        print(env_name + ': já disponível no ambiente')
    else:
        print(env_name + ': não configurado (opcional; sem ele o recurso correspondente fica desligado)')


In [ ]:
# 2. Opcional: montar Drive para persistir LoRAs, modelos e resultados
USAR_DRIVE = False
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['CLONFIELD_DIR'] = '/content/drive/MyDrive/ClonField_Lab'
else:
    import os
    os.environ['CLONFIELD_DIR'] = '/content/ClonField_Lab'
print('Pasta:', os.environ['CLONFIELD_DIR'])


In [ ]:
# Opcional: pré-baixar apenas versões Civitai escolhidas
# Deixe [] para abrir o Studio direto. Para usar, informe IDs de versão do assets.json.
VERSOES_CIVITAI = []
if VERSOES_CIVITAI:
    args = [sys.executable, "-u", str(repo / "cf_assets.py")]
    for version in VERSOES_CIVITAI:
        args.extend(["--version", str(version)])
    if subprocess.run(args).returncode:
        print("Alguns downloads ficaram pendentes. Você pode continuar para abrir o Studio.")
else:
    print("Pré-download pulado. Continue para iniciar o worker e o Studio.")


In [ ]:
# Iniciar o worker local de upscale. Os pesos são baixados no primeiro job.
START_COMFYUI = True
PREPARE_OPENJUTSU = False
PREPARE_SEETHROUGH = True  # clona ComfyUI-See-through (commit fixado) e instala dependências
PREPARE_FACEFUSION = False  # instala o FaceFusion (troca de rosto) em um venv próprio; ~1 GB de pesos no primeiro uso
PREPARE_GGUF = True  # clona ComfyUI-GGUF (commit fixado): necessário para o vídeo Wan 2.2 em GGUF (Q6_K)

if START_COMFYUI:
    import json
    import os
    import socket
    import subprocess
    import sys
    import time
    import urllib.error
    import urllib.request
    import venv
    from pathlib import Path

    revision = "v0.44.0"
    sys.path.insert(0, str(repo))
    import cf_tiers
    tier = cf_tiers.detect()
    print(f"GPU: {tier.gpu} · {tier.vram_gb:.0f} GB · perfil {tier.label}", flush=True)
    runtime = Path("/content/clonfield_runtime")
    checkout = runtime / f"ComfyUI-{revision}"
    python_env = runtime / f"comfy_venv-{revision}"
    models = runtime / "ComfyUI" / "models"
    log_path = runtime / "comfyui.log"
    os.environ["CLONFIELD_COMFY_LOG"] = str(log_path)
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))

    def comfy_has_seedvr2():
        try:
            for name in ("SeedVR2Preprocess", "SeedVR2TemporalChunk", "SeedVR2TemporalMerge"):
                with opener.open(f"http://127.0.0.1:8188/object_info/{name}", timeout=2) as response:
                    if name not in json.load(response):
                        return False
            return True
        except (OSError, ValueError, urllib.error.URLError):
            return False

    if comfy_has_seedvr2():
        if PREPARE_OPENJUTSU:
            raise RuntimeError("Pare o worker antes de instalar o pack OpenJutsu.")
        os.environ.setdefault("CLONFIELD_COMFY_MODELS", str(models))
        print("ComfyUI com SeedVR2 já está ativo.")
    else:
        with socket.socket() as port_check:
            port_check.settimeout(1)
            if port_check.connect_ex(("127.0.0.1", 8188)) == 0:
                raise RuntimeError("A porta 8188 tem um worker antigo. Reinicia a sessão do Colab e roda esta versão.")
        models.mkdir(parents=True, exist_ok=True)
        if not checkout.exists():
            subprocess.run(["git", "init", str(checkout)], check=True)
            subprocess.run(["git", "-C", str(checkout), "remote", "add", "origin",
                            "https://github.com/Comfy-Org/ComfyUI.git"], check=True)
        current = subprocess.run(["git", "-C", str(checkout), "rev-parse", "HEAD"],
                                 capture_output=True, text=True)
        if current.returncode:
            subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", "tag", revision], check=True)
            subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"], check=True)
        else:
            pinned_commit = subprocess.check_output(
                ["git", "-C", str(checkout), "rev-parse", f"{revision}^{{commit}}"], text=True
            ).strip()
            if current.stdout.strip() != pinned_commit:
                raise RuntimeError(f"O checkout dedicado diverge da versão {revision}.")

        executable = python_env / "bin" / "python"
        marker = python_env / "comfy_revision.txt"
        if not executable.is_file() or not marker.is_file() or marker.read_text().strip() != revision:
            print("Preparando o ambiente Python do ComfyUI...", flush=True)
            # Recria a configuração se a instalação anterior parou no ensurepip.
            venv.EnvBuilder(with_pip=False, system_site_packages=True).create(python_env)
            # Mantém o torch do Colab: reinstalar torch custa minutos e pode quebrar a GPU.
            frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
            pins = [line for line in frozen.splitlines()
                    if line.lower().split("==")[0] in ("torch", "torchvision", "torchaudio") and "==" in line]
            constraints = runtime / "torch-constraints.txt"
            constraints.write_text("\n".join(pins) + "\n")
            subprocess.run(
                [sys.executable, "-m", "pip", "--python", str(executable),
                 "install", "-r", str(checkout / "requirements.txt"), "-c", str(constraints)],
                check=True,
            )
            marker.write_text(revision, encoding="utf-8")
        os.environ["CLONFIELD_COMFY_MODELS"] = str(models)
        if PREPARE_OPENJUTSU:
            subprocess.run([sys.executable, str(repo / "prepare_openjutsu.py"), "--checkout", str(checkout),
                            "--models", str(models), "--python", str(executable)], check=True)
        if PREPARE_FACEFUSION:
            subprocess.run([sys.executable, str(repo / "prepare_facefusion.py"), os.environ.get("CLONFIELD_DIR", "/content/ClonField_Lab"), "--cuda"], check=True)
        if PREPARE_GGUF:
            subprocess.run([sys.executable, str(repo / "prepare_gguf.py"), str(checkout), "--pip", str(executable)], check=True)
        if PREPARE_SEETHROUGH:
            subprocess.run([sys.executable, str(repo / "prepare_seethrough.py"), str(checkout), "--pip", str(executable)], check=True)
        with log_path.open("w") as log:
            comfy_process = subprocess.Popen(
                [str(executable), "-u", "main.py", "--listen", "127.0.0.1", "--port", "8188", *tier.comfy_flags,
                 "--models-directory", str(models)],
                cwd=checkout, stdout=log, stderr=subprocess.STDOUT,
            )
        for _ in range(90):
            if comfy_has_seedvr2():
                print(f"ComfyUI pronto. Logs em {log_path}")
                break
            if comfy_process.poll() is not None:
                raise RuntimeError(log_path.read_text()[-5000:])
            time.sleep(1)
        else:
            comfy_process.terminate()
            raise RuntimeError("ComfyUI não ficou pronto em 90 s. " + log_path.read_text()[-3000:])


In [ ]:
# LLM local do chat e do agente (100% local, sem API externa). Roda no Colab; nada do que você escreve sai do runtime.
# Modelo: Qwen3.5-9B abliterado (sem filtro de recusa), GGUF Q6_K, servido pelo llama.cpp. Baixa ~7,4 GB + ~0,8 GB na primeira vez.
# NÃO validado em GPU ainda: o primeiro teste na T4 mede tempo de carga, VRAM e qualidade de chamada de ferramentas.
# Para liberar VRAM ao ComfyUI: llama.terminate() e depois cf_llama.stop(llama).
INICIAR_LLM_LOCAL = True
VARIANTE_LLM = "q6_k"  # q6_k (padrão) | q8_0 | q4_k_m
if INICIAR_LLM_LOCAL:
    import os, sys
    sys.path.insert(0, str(repo))
    import cf_llama
    lab = os.environ.get("CLONFIELD_DIR", "/content/ClonField_Lab")
    cf_llama.install_binary(lab, "cuda12")
    cf_llama.prepare(lab, VARIANTE_LLM)
    llama = cf_llama.start(lab, VARIANTE_LLM)
    print("LLM local pronto em", os.environ["CLONFIELD_LLAMA_URL"])


In [ ]:
# Iniciar o Studio com diagnóstico
from datetime import datetime, timezone
import os, signal

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
log_path = repo.parent / "logs" / f"diagnostico-{stamp}.jsonl"
args = [sys.executable, "-u", str(repo / "cf_diagnose.py"), "--log", str(log_path)]

print(f"Diagnóstico desta execução: {log_path}", flush=True)
process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, errors="replace", start_new_session=True)
public_ready = False
try:
    with process.stdout:
        for line in process.stdout:
            if line.startswith("CLONFIELD_READY "):
                port = int(line.split()[1])
                if public_ready:
                    print("Studio pronto. Use o link público e a senha para conectar o frontend do Studio. Mantenha esta célula rodando.", flush=True)
                else:
                    from google.colab import output
                    print("Studio pronto pelo Colab. Mantenha esta célula rodando.", flush=True)
                    output.serve_kernel_port_as_iframe(port, height=880)
            else:
                public_ready = public_ready or line.startswith("LINK PÚBLICO: ")
                print(line, end="", flush=True)
    status = process.wait()
except KeyboardInterrupt:
    print("Encerrando o Studio.", flush=True)
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
else:
    if status:
        comfy_log = repo.parent / "comfyui.log"
        if comfy_log.exists():
            print("--- últimas linhas do ComfyUI ---\n" + "\n".join(comfy_log.read_text(errors="replace").splitlines()[-30:]))
        raise RuntimeError(f"O Studio encerrou com código {status}. Consulte o diagnóstico em {log_path}.")
